In [30]:
import string
import os

class Libro:
    def __init__(self, filename: str):
        self.filename = filename

    def obtener_tokens(self) -> list[str]:
        """Abre el libro, lo pone en minusculas, quita puntuacion y regresa las palabras."""
        try:
            with open(self.filename, 'r', encoding='utf-8-sig') as archivo:
                contenido = archivo.read()
            contenido = contenido.lower()
            contenido = contenido.translate(str.maketrans('', '', string.punctuation))
            lista_de_palabras = contenido.split()
            return lista_de_palabras
        except FileNotFoundError:
            print(f"No encontré el archivo: {self.filename}")
            return []

In [31]:
carpeta_libros = 'Books/'
todos_los_tokens = {}
archivos_txt = [f for f in os.listdir(carpeta_libros) if f.endswith('.txt')]

for nombre_archivo in archivos_txt:
    ruta_completa = os.path.join(carpeta_libros, nombre_archivo)
    libro_objeto = Libro(ruta_completa)
    todos_los_tokens[nombre_archivo] = libro_objeto.obtener_tokens()

print(f"¡Listo! Limpiamos con éxito {len(todos_los_tokens)} libros.")

¡Listo! Limpiamos con éxito 32 libros.


In [32]:
def crear_vocabulario(tokens_diccionario):
    """Crea una lista única sin repeticiones de todas las palabras encontradas."""
    palabras_unicas = set()
    for lista_palabras in tokens_diccionario.values():
        palabras_unicas.update(lista_palabras)
    return sorted(list(palabras_unicas))

vocabulario_global = crear_vocabulario(todos_los_tokens)
print(f"Vocabulario total: {len(vocabulario_global)} palabras únicas.")

Vocabulario total: 156998 palabras únicas.


In [33]:
import math

def calcular_tf_idf(tokens_dict, vocabulario):
    """Calcula el peso TF-IDF para cada palabra en cada libro."""
    print("Calculando TF-IDF... Espera un momento...")

    total_libros = len(tokens_dict)

    conteos_por_libro = {}
    for nombre_libro, lista_tokens in tokens_dict.items():
        conteos = {}
        for palabra in lista_tokens:
            conteos[palabra] = conteos.get(palabra, 0) + 1
        conteos_por_libro[nombre_libro] = conteos

    libros_con_la_palabra = {}
    for nombre_libro, conteos in conteos_por_libro.items():
        for palabra in conteos:
            libros_con_la_palabra[palabra] = libros_con_la_palabra.get(palabra, 0) + 1

    matriz_tf_idf = {}
    for nombre_libro, conteos in conteos_por_libro.items():
        total_palabras = sum(conteos.values())
        vectores_libro = {}
        for palabra, veces in conteos.items():
            tf = veces / total_palabras
            idf = math.log(total_libros / (1 + libros_con_la_palabra[palabra]))
            vectores_libro[palabra] = tf * idf
        matriz_tf_idf[nombre_libro] = vectores_libro

    print("¡Problema 4 Completado con éxito!")
    return matriz_tf_idf

matriz_pesos = calcular_tf_idf(todos_los_tokens, vocabulario_global)

Calculando TF-IDF... Espera un momento...
¡Problema 4 Completado con éxito!


In [ ]:
class Clasificador:
    def __init__(self, matriz_tf_idf: dict) -> None:
        self._matriz = matriz_tf_idf

    @property
    def libros(self) -> list:
        return list(self._matriz.keys())

    def keys(self):
        """Regresa los nombres de los libros."""
        return self._matriz.keys()

    def values(self):
        """Regresa los vectores TF-IDF de cada libro."""
        return self._matriz.values()

    def items(self):
        """Regresa pares (nombre_libro, vector) de cada libro."""
        return self._matriz.items()

    def __getitem__(self, libro: str):
        """Permite acceder a un vector con clasificador[libro]."""
        return self._matriz[libro]

    def __len__(self):
        """Regresa el número de libros."""
        return len(self._matriz)

    def __contains__(self, libro: str):
        """Permite usar 'in' para checar si un libro existe."""
        return libro in self._matriz

    @staticmethod
    def _similitud_coseno(vec1: dict, vec2: dict) -> float:
        """Coseno del ángulo entre dos vectores TF-IDF."""
        dot = sum(value * vec2.get(palabra, 0.0) for palabra, value in vec1.items())
        mag1 = math.sqrt(sum(v**2 for v in vec1.values()))
        mag2 = math.sqrt(sum(v**2 for v in vec2.values()))
        if mag1 == 0 or mag2 == 0:
            return 0.0
        return dot / (mag1 * mag2)

    def recomendar(self, libro: str, top: int = 5) -> list:
        """Recomienda libros similares al dado."""
        vec_base = self._matriz[libro]
        resultados = []
        for otro, vec in self._matriz.items():
            if otro != libro:
                resultados.append((otro, self._similitud_coseno(vec_base, vec)))
        return sorted(resultados, key=lambda x: x[1], reverse=True)[:top]

    def resumen(self, libro: str, top: int = 10) -> None:
        """Muestra las palabras más representativas de un libro según TF-IDF."""
        vec = self._matriz[libro]
        palabras = sorted(vec.items(), key=lambda x: x[1], reverse=True)[:top]
        print(f"Palabras más representativas de: {libro}\n")
        for palabra, peso in palabras:
            print(f"  {peso:.4f} — {palabra}")

In [35]:
clasificador = Clasificador(matriz_pesos)

libro_favorito = archivos_txt[5]
print(f"Porque te gustó: {libro_favorito}\n")

print("Te recomendamos:")
for nombre, score in clasificador.recomendar(libro_favorito, top=10):
    print(f"  {score:.4f} — {nombre}")
print("\n--- Resumen del libro ---")
clasificador.resumen(libro_favorito)

Porque te gustó: Dracula by Bram Stoker (2047).txt

Te recomendamos:
  0.1745 — Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley (3942).txt
  0.1706 — A Room with a View by E. M.  Forster (2907).txt
  0.1569 — The Expedition of Humphry Clinker by T.  Smollett (1426).txt
  0.1534 — The City of God, Volume I by Saint of Hippo Augustine (7070).txt
  0.1468 — The Adventures of Ferdinand Count Fathom — Complete by T.  Smollett (1427).txt
  0.1419 — Jane Eyre_ An Autobiography by Charlotte Brontë (2170).txt
  0.1211 — My Life — Volume 1 by Richard Wagner (2031).txt
  0.1141 — The King in Yellow by Robert W.  Chambers (1884).txt
  0.1035 — The Confessions of St. Augustine by Saint of Hippo Augustine (1510).txt
  0.1029 — The Adventures of Sherlock Holmes by Arthur Conan Doyle (1942).txt

--- Resumen del libro ---


AttributeError: 'Clasificador' object has no attribute 'resumen'